#### We have two datasets:
- Sessions Table: Contains records of when users started their sessions.
- Order Summary Table: Contains records of orders placed by users along with their values.

#### We want to:
- Find users who started a session and placed an order on the same day.
- Calculate the total number of orders and the total order value for those users.

#### Solution
- We will convert the session and order dates to just the date (ignoring time) using the to_date() function. This step ensures that we can match sessions and orders that happened on the same calendar day, which is the core of our analysis.

- Finally, we’ll join the two DataFrames on user_id and matching dates. After grouping the data by user_id and session date, we’ll calculate the total number of orders and total order value for each user on the same day. We’ll filter out users who didn’t place any orders, and then display the results.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, count, sum

In [0]:
# Initialize Spark session
spark = SparkSession.builder.appName("SessionOrderAnalysis").getOrCreate()

In [0]:
# Sample data for sessions
sessions_data = [
    (1, 1, '2024-01-01 00:00:00'),
    (2, 2, '2024-01-02 00:00:00'),
    (3, 3, '2024-01-05 00:00:00'),
    (4, 3, '2024-01-05 00:00:00'),
    (5, 4, '2024-01-03 00:00:00'),
    (6, 4, '2024-01-03 00:00:00'),
    (7, 5, '2024-01-04 00:00:00'),
    (8, 5, '2024-01-04 00:00:00'),
    (9, 3, '2024-01-05 00:00:00'),
    (10, 5, '2024-01-04 00:00:00')
]

In [0]:
# Sample data for orders
orders_data = [
    (1, 1, 152, '2024-01-01 00:00:00'),
    (2, 2, 485, '2024-01-02 00:00:00'),
    (3, 3, 398, '2024-01-05 00:00:00'),
    (4, 3, 320, '2024-01-05 00:00:00'),
    (5, 4, 156, '2024-01-03 00:00:00'),
    (6, 4, 121, '2024-01-03 00:00:00'),
    (7, 5, 238, '2024-01-04 00:00:00'),
    (8, 5, 70, '2024-01-04 00:00:00'),
    (9, 3, 152, '2024-01-05 00:00:00'),
    (10, 5, 171, '2024-01-04 00:00:00')
]

In [0]:
session_columns = ["session_id", "user_id", "session_date"]
orders_columns = ["order_id", "user_id", "order_value", "order_date"]

In [0]:
# Convert data into DataFrames
sessions_df = spark.createDataFrame(sessions_data, session_columns)
orders_df = spark.createDataFrame(orders_data, orders_columns)

In [0]:
# Convert session_date and order_date to date format (ignoring time part)
sessions_df = sessions_df.withColumn('session_date_only', to_date(col('session_date')))
orders_df = orders_df.withColumn('order_date_only', to_date(col('order_date')))

In [0]:
# Join sessions and orders on user_id and matching session_date and order_date
joined_df = sessions_df.alias('s').join(orders_df.alias('o'), 
                                        (col('s.user_id') == col('o.user_id')) & 
                                        (col('s.session_date_only') == col('o.order_date_only')), 
                                        'inner')

In [0]:
# Group by user_id and session_date, and calculate the total number of orders and total value of orders
result_df = joined_df.groupBy('s.user_id', 's.session_date_only') \
    .agg(count('o.order_id').alias('total_orders'), 
         sum('o.order_value').alias('total_order_value'))

In [0]:
# Filter the result to only show users who placed at least one order
result_df = result_df.filter(col('total_orders') > 0)


In [0]:
# Show the final result
result_df.show(truncate=False)

+-------+-----------------+------------+-----------------+
|user_id|session_date_only|total_orders|total_order_value|
+-------+-----------------+------------+-----------------+
|1      |2024-01-01       |1           |152              |
|2      |2024-01-02       |1           |485              |
|3      |2024-01-05       |9           |2610             |
|4      |2024-01-03       |4           |554              |
|5      |2024-01-04       |9           |1437             |
+-------+-----------------+------------+-----------------+

